Infrastructure Snowflake et ingestion des données structurées pour le projet ShopFlow.
*Co-authored with CoCo*

## 1. Création de la base de données SHOPFLOW_DB

On crée une base dédiée au projet avec trois schémas séparés par responsabilité :
- **RAW** : données brutes telles qu'ingérées (zone d'atterrissage)
- **STAGING** : données nettoyées, typées, dédupliquées
- **MARTS** : modèles analytiques prêts à consommer

Cette séparation en couches (medallion architecture) permet de tracer la provenance des données et d'isoler les transformations.

In [ ]:
%%sql -r res_create_db
CREATE DATABASE IF NOT EXISTS SHOPFLOW_DB;

In [ ]:
%%sql -r res_create_schemas
CREATE SCHEMA IF NOT EXISTS SHOPFLOW_DB.RAW;
CREATE SCHEMA IF NOT EXISTS SHOPFLOW_DB.STAGING;
CREATE SCHEMA IF NOT EXISTS SHOPFLOW_DB.MARTS;

## 2. Création des Virtual Warehouses

Deux warehouses distincts pour séparer les charges de travail :
- **WH_INGEST** (XSMALL) : dédié aux opérations COPY INTO — peu gourmandes en compute, on veut juste déplacer des fichiers vers les tables. Auto-suspend à 60s pour couper les coûts dès l'inactivité.
- **WH_TRANSFORM** (SMALL) : dédié aux Tasks de transformation (jointures, agrégations). Un peu plus de puissance, même politique d'auto-suspend.

Séparer les warehouses évite qu'une grosse transformation bloque l'ingestion et facilite le suivi des coûts par workload.

In [ ]:
%%sql -r res_wh_ingest
CREATE WAREHOUSE IF NOT EXISTS WH_INGEST
  WAREHOUSE_SIZE = 'XSMALL'
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE
  INITIALLY_SUSPENDED = TRUE
  COMMENT = 'Warehouse dédié aux opérations COPY INTO';

In [ ]:
%%sql -r res_wh_transform
CREATE WAREHOUSE IF NOT EXISTS WH_TRANSFORM
  WAREHOUSE_SIZE = 'SMALL'
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE
  INITIALLY_SUSPENDED = TRUE
  COMMENT = 'Warehouse dédié aux Tasks de transformation';

## 3. Création du rôle applicatif SHOPFLOW_ENGINEER

On crée un rôle dédié au projet plutôt que d'utiliser SYSADMIN directement. Avantages :
- **Principe du moindre privilège** : ce rôle n'a accès qu'aux ressources du projet
- **Auditabilité** : on sait qui fait quoi via ce rôle
- **Collaboration** : on peut l'attribuer à plusieurs utilisateurs sans donner des droits admin

Les grants couvrent : usage des warehouses, ownership sur la base et ses schémas, et capacité de créer des stages/tables.

In [ ]:
%%sql -r res_role_grants
-- Utiliser USERADMIN pour créer le rôle (SYSADMIN n'a pas CREATE ROLE)
USE ROLE USERADMIN;

CREATE ROLE IF NOT EXISTS SHOPFLOW_ENGINEER
  COMMENT = 'Rôle applicatif pour le projet ShopFlow';

-- Attribuer le rôle à SYSADMIN (hiérarchie)
GRANT ROLE SHOPFLOW_ENGINEER TO ROLE SYSADMIN;

-- Revenir à SYSADMIN pour les grants sur les objets qu'il possède
USE ROLE SYSADMIN;

-- Grants sur les warehouses
GRANT USAGE ON WAREHOUSE WH_INGEST TO ROLE SHOPFLOW_ENGINEER;
GRANT USAGE ON WAREHOUSE WH_TRANSFORM TO ROLE SHOPFLOW_ENGINEER;

-- Grants sur la base et les schémas
GRANT USAGE ON DATABASE SHOPFLOW_DB TO ROLE SHOPFLOW_ENGINEER;
GRANT ALL PRIVILEGES ON SCHEMA SHOPFLOW_DB.RAW TO ROLE SHOPFLOW_ENGINEER;
GRANT ALL PRIVILEGES ON SCHEMA SHOPFLOW_DB.STAGING TO ROLE SHOPFLOW_ENGINEER;
GRANT ALL PRIVILEGES ON SCHEMA SHOPFLOW_DB.MARTS TO ROLE SHOPFLOW_ENGINEER;

-- Grants sur les tables existantes
GRANT ALL PRIVILEGES ON ALL TABLES IN SCHEMA SHOPFLOW_DB.RAW TO ROLE SHOPFLOW_ENGINEER;
GRANT ALL PRIVILEGES ON ALL TABLES IN SCHEMA SHOPFLOW_DB.STAGING TO ROLE SHOPFLOW_ENGINEER;
GRANT ALL PRIVILEGES ON ALL TABLES IN SCHEMA SHOPFLOW_DB.MARTS TO ROLE SHOPFLOW_ENGINEER;

## 4. Création du Stage interne

Un **stage interne** est une zone de stockage temporaire dans Snowflake où l'on dépose les fichiers avant de les charger dans les tables. C'est la zone d'atterrissage (landing zone).

On le crée dans le schéma RAW car c'est là qu'arrivent les données brutes.

In [ ]:
%%sql -r res_stage
CREATE STAGE IF NOT EXISTS SHOPFLOW_DB.RAW.STAGE_LANDING
  COMMENT = 'Stage interne pour déposer les fichiers CSV et Parquet avant ingestion';

## 5. Upload des fichiers de données

Pour charger les fichiers dans le stage, on utilise la commande `PUT` :
- **orders.csv** : commandes clients
- **order_items.csv** : lignes de détail des commandes
- **customers.parquet** : référentiel clients au format Parquet (colonnes typées, compression native)

> **Note** : La commande PUT fonctionne depuis SnowSQL ou via l'UI Snowsight (bouton Upload dans le stage). Dans un notebook, on peut utiliser `PUT` directement si les fichiers sont accessibles depuis le workspace.

In [ ]:
%%sql -r res_list_stage
-- Upload des fichiers locaux vers le stage interne
--PUT 'file://C:\Users\marel\Desktop\Planus_Stage\shopflow-apprenants\data\j1\customers.parquet' @SHOPFLOW_DB.RAW.STAGE_LANDING/customers AUTO_COMPRESS=FALSE;
--PUT 'file://C:\Users\marel\Desktop\Planus_Stage\shopflow-apprenants\data\j1\orders.csv' @SHOPFLOW_DB.RAW.STAGE_LANDING/orders AUTO_COMPRESS=TRUE;
--PUT 'file://C:\Users\marel\Desktop\Planus_Stage\shopflow-apprenants\data\j1\order_items.csv' @SHOPFLOW_DB.RAW.STAGE_LANDING/order_items AUTO_COMPRESS=TRUE;
--PUT 'file://C:\Users\marel\Desktop\Planus_Stage\shopflow-apprenants\data\j1\products.json' @SHOPFLOW_DB.RAW.STAGE_LANDING/products AUTO_COMPRESS=TRUE;
--PUT 'file://C:\Users\marel\Desktop\Planus_Stage\shopflow-apprenants\data\j1\web_events.json' @SHOPFLOW_DB.RAW.STAGE_LANDING/web_events AUTO_COMPRESS=TRUE;

-- Vérifier le contenu du stage
LIST @SHOPFLOW_DB.RAW.STAGE_LANDING;

## 6. Définition des File Formats

Les **file formats** décrivent la structure des fichiers à charger. Snowflake a besoin de savoir :
- Le type de fichier (CSV, Parquet, JSON...)
- Le délimiteur, l'encodage, la gestion des guillemets (pour CSV)
- Si la première ligne est un header

On crée deux formats réutilisables :
- **FF_CSV_ORDERS** : pour nos fichiers CSV (virgule, header, guillemets doubles)
- **FF_PARQUET** : pour le fichier Parquet (Snowflake gère nativement le schéma Parquet)

In [ ]:
%%sql -r res_ff_csv
CREATE FILE FORMAT IF NOT EXISTS SHOPFLOW_DB.RAW.FF_CSV_ORDERS
  TYPE = 'CSV'
  FIELD_DELIMITER = ','
  SKIP_HEADER = 1
  FIELD_OPTIONALLY_ENCLOSED_BY = '"'
  NULL_IF = ('', 'NULL', 'null')
  EMPTY_FIELD_AS_NULL = TRUE
  COMMENT = 'Format CSV pour orders et order_items';

In [ ]:
%%sql -r res_ff_parquet
CREATE FILE FORMAT IF NOT EXISTS SHOPFLOW_DB.RAW.FF_PARQUET
  TYPE = 'PARQUET'
  COMPRESSION = 'SNAPPY'
  COMMENT = 'Format Parquet pour customers';

## 7. Création des tables cibles dans RAW

On crée les tables qui vont recevoir les données brutes. La structure reflète exactement les colonnes des fichiers sources :
- **RAW_ORDERS** : une ligne par commande
- **RAW_ORDER_ITEMS** : une ligne par article commandé
- **RAW_CUSTOMERS** : une ligne par client

On ajoute une colonne `_LOADED_AT` (timestamp d'ingestion) pour la traçabilité.

In [ ]:
%%sql -r res_tbl_orders
CREATE TABLE IF NOT EXISTS SHOPFLOW_DB.RAW.RAW_ORDERS (
  ORDER_ID        VARCHAR,
  CUSTOMER_ID     VARCHAR,
  ORDER_DATE      VARCHAR,
  STATUS          VARCHAR,
  TOTAL_AMOUNT    VARCHAR,
  _LOADED_AT      TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
)
COMMENT = 'Commandes brutes ingérées depuis orders.csv';

In [ ]:
%%sql -r res_tbl_items
CREATE TABLE IF NOT EXISTS SHOPFLOW_DB.RAW.RAW_ORDER_ITEMS (
  ORDER_ITEM_ID   VARCHAR,
  ORDER_ID        VARCHAR,
  PRODUCT_ID      VARCHAR,
  QUANTITY        VARCHAR,
  UNIT_PRICE      VARCHAR,
  _LOADED_AT      TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
)
COMMENT = 'Lignes de commande brutes ingérées depuis order_items.csv';

In [ ]:
%%sql -r res_tbl_customers
CREATE TABLE IF NOT EXISTS SHOPFLOW_DB.RAW.RAW_CUSTOMERS (
  CUSTOMER_ID     VARCHAR,
  FIRST_NAME      VARCHAR,
  LAST_NAME       VARCHAR,
  EMAIL           VARCHAR,
  CITY            VARCHAR,
  COUNTRY         VARCHAR,
  CREATED_AT      VARCHAR,
  _LOADED_AT      TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
)
COMMENT = 'Clients bruts ingérés depuis customers.parquet';

## 8. Ingestion avec COPY INTO

La commande `COPY INTO` charge les données du stage vers les tables cibles :
- Elle est **idempotente** : les fichiers déjà chargés ne sont pas rechargés (grâce aux métadonnées de chargement)
- Elle est **parallélisée** : Snowflake découpe les fichiers et les charge en parallèle
- On utilise le warehouse WH_INGEST pour ces opérations

> Si les fichiers ne sont pas encore dans le stage, décommentez les commandes PUT de l'étape 5 ou uploadez-les via l'UI.

In [ ]:
%%sql -r res_use_wh
USE WAREHOUSE WH_INGEST;

In [ ]:
%%sql -r res_copy_orders
COPY INTO SHOPFLOW_DB.RAW.RAW_ORDERS (ORDER_ID, CUSTOMER_ID, ORDER_DATE, STATUS, TOTAL_AMOUNT)

FROM @SHOPFLOW_DB.RAW.STAGE_LANDING/orders
FILE_FORMAT = (FORMAT_NAME = 'SHOPFLOW_DB.RAW.FF_CSV_ORDERS')
ON_ERROR = 'CONTINUE';

In [ ]:
%%sql -r res_copy_items
COPY INTO SHOPFLOW_DB.RAW.RAW_ORDER_ITEMS(ORDER_ID,PRODUCT_ID,QUANTITY,UNIT_PRICE)
FROM @SHOPFLOW_DB.RAW.STAGE_LANDING/order_items
FILE_FORMAT = (FORMAT_NAME = 'SHOPFLOW_DB.RAW.FF_CSV_ORDERS')
ON_ERROR = 'CONTINUE';

In [ ]:
%%sql -r res_copy_customers
COPY INTO SHOPFLOW_DB.RAW.RAW_CUSTOMERS
FROM @SHOPFLOW_DB.RAW.STAGE_LANDING/customers
FILE_FORMAT = (FORMAT_NAME = 'SHOPFLOW_DB.RAW.FF_PARQUET')
MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE
ON_ERROR = 'CONTINUE';

## 9. Vérification de l'ingestion

On vérifie que les données sont bien arrivées :
1. **COUNT(*)** sur chaque table pour confirmer le volume
2. **VALIDATION_MODE** pour détecter les lignes rejetées lors du COPY

Si des erreurs apparaissent, on peut consulter `COPY_HISTORY` dans `INFORMATION_SCHEMA` pour le détail.

In [ ]:
%%sql -r res_counts
SELECT 'RAW_ORDERS' AS table_name, COUNT(*) AS row_count FROM SHOPFLOW_DB.RAW.RAW_ORDERS
UNION ALL
SELECT 'RAW_ORDER_ITEMS', COUNT(*) FROM SHOPFLOW_DB.RAW.RAW_ORDER_ITEMS
UNION ALL
SELECT 'RAW_CUSTOMERS', COUNT(*) FROM SHOPFLOW_DB.RAW.RAW_CUSTOMERS;

In [ ]:
%%sql -r res_validation
-- Validation mode : simule le COPY sans charger, pour voir les erreurs potentielles
-- Décommenter et adapter si besoin de diagnostiquer des erreurs

-- COPY INTO SHOPFLOW_DB.RAW.RAW_ORDERS
-- FROM @SHOPFLOW_DB.RAW.STAGE_LANDING/orders
-- FILE_FORMAT = (FORMAT_NAME = 'SHOPFLOW_DB.RAW.FF_CSV_ORDERS')
-- VALIDATION_MODE = 'RETURN_ERRORS';

-- Historique de chargement
SELECT *
FROM TABLE(INFORMATION_SCHEMA.COPY_HISTORY(
  TABLE_NAME => 'SHOPFLOW_DB.RAW.RAW_ORDERS',
  START_TIME => DATEADD(HOUR, -1, CURRENT_TIMESTAMP())
));

## Résumé

| Étape | Objet créé | Rôle |
|-------|-----------|------|
| 1 | `SHOPFLOW_DB` + 3 schémas | Architecture medallion |
| 2 | `WH_INGEST`, `WH_TRANSFORM` | Séparation des workloads |
| 3 | `SHOPFLOW_ENGINEER` | Sécurité & moindre privilège |
| 4 | `STAGE_LANDING` | Zone d'atterrissage fichiers |
| 5 | PUT / Upload | Fichiers → Stage |
| 6 | `FF_CSV_ORDERS`, `FF_PARQUET` | Déclaration des formats |
| 7 | 3 tables RAW_* | Cibles d'ingestion |
| 8 | COPY INTO | Chargement effectif |
| 9 | COUNT + VALIDATION | Contrôle qualité |